# MambaGuard — GPU study on Kaggle (version 3)

One background run (`STUDY = "all"`, about 4–5 h on a T4) produces `results_v3_all.zip`:

* **main**: flat-read-out MambaGuard with chunk scoring (the variant chosen on validation in v2) × 5 benchmarks × 5 seeds;
  for every run the lexical (TF-IDF) and label-free novelty (Mahalanobis) experts are scored on the same episodes;
  validation is split by group into a *selection* half and a *conformal-calibration* half; certificates and latency.
* **heldout**: the same on the held-out-template folds × 3 seeds (A2A: 1 seed).

Cell 9 runs the pre-registered analysis (`scripts/analyse_v3.py`): hybrid of the three experts with weights chosen on
validation, split-conformal and adaptive-conformal false-alarm control, paired bootstrap against the lexical baseline.

**Before running:** Settings → *Accelerator* = **GPU T4 x2**, *Internet* = **On**, notebook **Private**.
Run cells 1–7 by hand (cell 7 = ~5-minute smoke test, must show no `ERROR`), then
**Save Version → Save & Run All (Commit)** so the run survives closing the browser.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- set Accelerator to GPU T4 x2")
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f"compute capability {major}.{minor}; bf16 supported: {torch.cuda.is_bf16_supported()}")
    if major < 7:
        print("WARNING: this GPU (e.g. P100) may not be supported by Kaggle's current PyTorch build. Switch to T4 x2.")

## 2. Get the code

Option A (default): clone the public GitHub branch (needs Internet).
Option B: upload `mambaguard-kaggle.zip` as a Kaggle *Dataset* and attach it with **+ Add Input**; this cell finds it
automatically under `/kaggle/input` and uses it instead of GitHub.

In [ ]:
import os, glob, subprocess, shutil
REPO_URL = "https://github.com/rogerpanel/CV.git"
BRANCH   = "claude/implement-mambaguard-detection-zCgu6"
CODE = None
for init in glob.glob("/kaggle/input/**/mambaguard/__init__.py", recursive=True):
    src = os.path.dirname(os.path.dirname(init))
    CODE = "/kaggle/working/mambaguard_code"          # copy: /kaggle/input is read-only
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE, ignore=shutil.ignore_patterns("__pycache__"))
    print("using uploaded dataset:", src)
    break
if CODE is None:
    shutil.rmtree("/tmp/mg_repo", ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, "--filter=blob:none",
                    "--sparse", REPO_URL, "/tmp/mg_repo"], check=True)
    subprocess.run(["git", "-C", "/tmp/mg_repo", "sparse-checkout", "set", "MambaGuard"], check=True)
    CODE = "/tmp/mg_repo/MambaGuard"
    print("cloned", BRANCH, "commit", subprocess.check_output(["git", "-C", "/tmp/mg_repo", "rev-parse", "--short", "HEAD"], text=True).strip())
os.chdir(CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, CODE)
print("code at", os.getcwd()); print(sorted(os.listdir(".")))

## 3. Install the two extra packages (torch is already installed — do not reinstall it)

In [ ]:
!pip install -q -r kaggle/requirements-kaggle.txt
import einops, sentence_transformers; print("einops", einops.__version__, "| sentence-transformers", sentence_transformers.__version__)

## 4. Download the benchmarks at their pinned commits

Saved to `/tmp/mambaguard_data` (not `/kaggle/working`, so they are not part of the output: MCPTox and
MCP-SafetyBench have no licence and must not be redistributed). AgentDojo is the largest (~0.5 GB, 1–3 min).
The A2A sandbox is generated, no download.

In [ ]:
DATA = "/tmp/mambaguard_data"
!bash kaggle/setup_benchmarks.sh {DATA} mcptox mcp_safetybench agentdojo injecagent

## 5. Check the loaders (episode counts should match `reproducibility/PROTOCOL_BENCHMARKS.md`)

In [ ]:
from mambaguard.data.protocol_traces import EPISODE_LOADERS
for name, folder in [("mcptox", "MCPTox-Benchmark"), ("mcp_safetybench", "MCPSafety"),
                     ("agentdojo", "agentdojo"), ("injecagent", "InjecAgent"), ("a2a_sandbox", "builtin")]:
    root = folder if folder == "builtin" else f"{DATA}/{folder}"
    eps = EPISODE_LOADERS[name](root, "all")
    att = sum(e.is_attack for e in eps)
    print(f"{name:16s} episodes={len(eps):5d}  attack={att:5d}  benign={len(eps) - att:5d}")

## 6. Pre-download the MiniLM sentence encoder (~90 MB)

In [ ]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
print("embedding dim", m.get_sentence_embedding_dimension())
del m

## 7. Smoke test (~5 min): tiny data, 2 epochs, both studies. Must finish with no `ERROR` lines (its numbers are meaningless).

In [ ]:
!python kaggle/run_kaggle_study.py --smoke --study all --data-root {DATA} --out /kaggle/working/results_v3_all
!python scripts/analyse_v3.py /kaggle/working/results_v3_all_smoke.zip | head -40

## 8. The real run — keep `STUDY = "all"`, then Save Version → Save & Run All (Commit)

Estimated 4–5 h on a T4 (the runner stops starting new jobs after 11 h and zips what it has).
To split it into two sessions instead: `STUDY = "main"`, then a second commit with `STUDY = "heldout"`.
If a session times out: attach that version's output (**+ Add Input → Notebook Output**), set `RESUME` to its
`results_v3_<study>` folder and commit again with the same `STUDY`.

In [ ]:
STUDY  = "all"    # or "main", then "heldout"
RESUME = ""       # e.g. "/kaggle/input/mambaguard-gpu-study/results_v3_all"  (only after a timeout)
EXTRA  = ""       # e.g. "--seeds 0 1 2" to shorten
resume_flag = f"--resume-from {RESUME}" if RESUME else ""
!python kaggle/run_kaggle_study.py --study {STUDY} --data-root {DATA} --out /kaggle/working/results_v3_{STUDY} --time-budget-h 11 {resume_flag} {EXTRA}

## 9. Package, summarise and download

Self-contained: it works after a kernel restart and also packages a results folder whose run was interrupted
(each finished job was already saved as a JSON file). Model weights are left out of the zip.

In [ ]:
import glob, os, subprocess, sys
os.chdir("/kaggle/working")
print(subprocess.run("ls -la /kaggle/working", shell=True, capture_output=True, text=True).stdout)
for d in sorted(glob.glob("/kaggle/working/results_*")):
    if os.path.isdir(d) and not os.path.exists(d + ".zip"):          # interrupted run: zip what is there
        n = len(glob.glob(d + "/runs/*.json"))
        name = os.path.basename(d)
        subprocess.run(f"zip -qr {name}.zip {name} -x '{name}/checkpoints/*'", shell=True, check=True)
        print(f"packaged {name}.zip ({n} finished jobs)")
zips = sorted(z for z in glob.glob("/kaggle/working/results_*.zip") if "checkpoints" not in z and "smoke" not in z)
zips += sorted(glob.glob("/kaggle/input/**/results_*.zip", recursive=True))   # earlier versions attached as input
print(zips)
CODE = next((p for p in ["/tmp/mg_repo/MambaGuard", "/kaggle/working/mambaguard_code"] if os.path.isdir(p)), None)
if CODE and zips:
    print(subprocess.run([sys.executable, f"{CODE}/scripts/summarise_kaggle_results.py", *zips],
                         capture_output=True, text=True).stdout[:20000])
    v3 = [z for z in zips if "results_v3_" in z]
    if v3:   # pre-registered hybrid / conformal / paired-test analysis
        print(subprocess.run([sys.executable, f"{CODE}/scripts/analyse_v3.py", *v3],
                             capture_output=True, text=True).stdout[:20000])
from IPython.display import FileLink, display
for z in zips:
    if z.startswith("/kaggle/working/"):
        display(FileLink(os.path.basename(z)))
# Or: right-hand panel -> Output (/kaggle/working) -> the zip -> three dots -> Download